# TW Stock Aggressive Fill Strategy

Set `SYMBOL`, `START_DATE`, `END_DATE`, `START_TIME`, and `END_TIME` in the next cell. The notebook calls `scripts.tw_stock_data_to_npz.convert_tw_stock_to_npz()` to generate the npz data file before running the strategy.


In [8]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.tw_stock_data_to_npz import convert_tw_stock_to_npz
from scripts.tw_stock_hftbacktest import (
    BacktestConfig,
    build_backtest,
    close_backtest,
    import_hftbacktest,
    print_state,
    state_snapshot,
    submit_limit_order,
    wait_for_bbo,
)

SYMBOL = "2330"
START_DATE = "2025-09-09"
END_DATE = START_DATE
START_TIME = None  # e.g. "09:00:00"
END_TIME = None    # e.g. "13:30:00"

DATA_FILE, event_data = convert_tw_stock_to_npz(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    start_time=START_TIME,
    end_time=END_TIME,
    workspace_root=ROOT,
)

hbtpkg = import_hftbacktest(ROOT)
CONFIG = BacktestConfig(data=DATA_FILE, order_latency_ns=0)
DATA_FILE


2026-07-01 10:20:10,379 - INFO - Found 5 parquet files
2026-07-01 10:20:10,380 - INFO - Filtering for date range 20250909 to 20250909


input_rows=46852
converted_rows=46852
skipped_symbol_rows=0
skipped_status_rows=0
skipped_time_rows=0
raw_events=570212
output_events=570212
depth_events=562224
trade_events=7988
opening_jump_qty=2819.0
first_exch_ts=1757377804629985000
last_exch_ts=1757395800000000000
min_feed_latency=0
max_feed_latency=0
qa_rows_checked=1000
best_bid_mismatches=0
best_ask_mismatches=0
trade_qty_mismatches=0
output=C:\Users\zoufuc\Desktop\hftbacktest\data\tw_stock_events\2330_20250909.npz


WindowsPath('C:/Users/zoufuc/Desktop/hftbacktest/data/tw_stock_events/2330_20250909.npz')

In [9]:
def run_aggressive_fill_strategy(hbt, hbtpkg, qty=1.0, round_trips=1, response_timeout_ns=10_000_000):
    asset_no = 0
    wait_for_bbo(hbt, asset_no)
    print_state("initial_bbo", None, state_snapshot(hbt, asset_no, CONFIG.contract_size))

    order_id = 10_001
    for round_no in range(1, round_trips + 1):
        depth = hbt.depth(asset_no)
        print(f"\nround={round_no} aggressive buy at best ask")
        before = state_snapshot(hbt, asset_no, CONFIG.contract_size)
        print_state("before_buy", order_id, before)
        rc = submit_limit_order(hbt, hbtpkg, asset_no, order_id, "buy", float(depth.best_ask), qty)
        response = hbt.wait_order_response(asset_no, order_id, response_timeout_ns)
        print(f"submit_buy order_id={order_id} rc={rc} response={response}")
        print_state("after_buy", order_id, state_snapshot(hbt, asset_no, CONFIG.contract_size))
        hbt.clear_inactive_orders(asset_no)
        order_id += 1

        depth = hbt.depth(asset_no)
        print(f"\nround={round_no} aggressive sell at best bid")
        before = state_snapshot(hbt, asset_no, CONFIG.contract_size)
        print_state("before_sell", order_id, before)
        rc = submit_limit_order(hbt, hbtpkg, asset_no, order_id, "sell", float(depth.best_bid), qty)
        response = hbt.wait_order_response(asset_no, order_id, response_timeout_ns)
        print(f"submit_sell order_id={order_id} rc={rc} response={response}")
        print_state("after_sell", order_id, state_snapshot(hbt, asset_no, CONFIG.contract_size))
        hbt.clear_inactive_orders(asset_no)
        order_id += 1

    print("\nfinal")
    print_state("final_state", None, state_snapshot(hbt, asset_no, CONFIG.contract_size))


In [10]:
hbt = build_backtest(CONFIG, hbtpkg)
try:
    run_aggressive_fill_strategy(hbt, hbtpkg, qty=1.0, round_trips=1)
finally:
    close_backtest(hbt)

initial_bbo                      bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=0.00 fee=0.00 equity=0.00 trades=0 value=0.00 volume=0.0000

round=1 aggressive buy at best ask
before_buy         order_id=10001 bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=0.00 fee=0.00 equity=0.00 trades=0 value=0.00 volume=0.0000
submit_buy order_id=10001 rc=0 response=0
after_buy          order_id=10001 bid=1190.00 ask=1195.00 mark=1192.50 pos=1.0000 balance=-1195000.00 fee=0.00 equity=-2500.00 trades=1 value=1195000.00 volume=1.0000

round=1 aggressive sell at best bid
before_sell        order_id=10002 bid=1190.00 ask=1195.00 mark=1192.50 pos=1.0000 balance=-1195000.00 fee=0.00 equity=-2500.00 trades=1 value=1195000.00 volume=1.0000
submit_sell order_id=10002 rc=0 response=0
after_sell         order_id=10002 bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=-5000.00 fee=0.00 equity=-5000.00 trades=2 value=2385000.00 volume=2.0000

final
final_state                      bid=1